# Clase 4 · Notebook 01
# Clasificador binario y métricas de evaluación

**Introducción al Deep Learning — Módulo 2, Unidad 2: Clasificación**

Vamos a construir una **red de neuronas para clasificación binaria** con Keras y, sobre todo, a **evaluarla
bien**: matriz de confusión, accuracy, precision, recall, especificidad y F1-score.

Usaremos el dataset de **cáncer de mama** (maligno / benigno).

1. Preparar los datos.
2. Construir la red (salida sigmoide + binary crossentropy).
3. Entrenar.
4. Evaluar con la matriz de confusión y todas las métricas.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Preparar los datos

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
tf.random.set_seed(42); np.random.seed(42)

data = load_breast_cancer()
X, y = data.data, data.target      # y: 0 = maligno, 1 = benigno

# Normalizamos (z-score) y dividimos en train/test de forma estratificada
X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Entrenamiento:", X_train.shape, "| Test:", X_test.shape)
print("Atributos:", X.shape[1])

## 2. Construir la red

Para clasificación **binaria**: una capa oculta ReLU y una capa de salida con **1 neurona sigmoide**
(devuelve la probabilidad de la clase 1). Pérdida: **binary crossentropy**.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

modelo = Sequential([
    Input(shape=(X.shape[1],)),
    Dense(18, activation="relu"),
    Dense(1, activation="sigmoid"),
])
modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo.summary()

## 3. Entrenar

In [ ]:
historia = modelo.fit(X_train, y_train, epochs=50, batch_size=16,
                      validation_split=0.1, verbose=0)
print("Entrenamiento terminado.")
print("Accuracy en test: %.1f %%" % (modelo.evaluate(X_test, y_test, verbose=0)[1] * 100))

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(historia.history["accuracy"], label="entrenamiento", color="#000F9F")
plt.plot(historia.history["val_accuracy"], label="validación", color="#FF647E")
plt.title("Exactitud durante el entrenamiento"); plt.xlabel("Época"); plt.ylabel("Accuracy")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 4. Evaluación: matriz de confusión y métricas

Pasamos las probabilidades a clases con un umbral de 0,5 y construimos la **matriz de confusión**.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

probs = modelo.predict(X_test, verbose=0).ravel()
y_pred = (probs >= 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=data.target_names).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Matriz de confusión"); plt.tight_layout(); plt.show()
print(cm)

### Métricas calculadas a mano (a partir de la matriz)

Tomamos la clase **maligno** como positiva para el cálculo manual.

In [ ]:
# Con etiquetas 0=maligno, 1=benigno, definimos "maligno" como positivo
tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[1, 0]).ravel()
# (labels=[1,0] -> positivo = 0 = maligno)

accuracy    = (tp + tn) / (tp + tn + fp + fn)
precision   = tp / (tp + fp)
recall      = tp / (tp + fn)          # sensibilidad / TPR
especificid = tn / (tn + fp)          # TNR
f1          = 2 * precision * recall / (precision + recall)

print("Positivo = 'maligno'\n")
print(f"VP={tp}  VN={tn}  FP={fp}  FN={fn}\n")
print(f"Accuracy      : {accuracy:.3f}")
print(f"Precision     : {precision:.3f}")
print(f"Recall        : {recall:.3f}")
print(f"Especificidad : {especificid:.3f}")
print(f"F1-Score      : {f1:.3f}")

### Las mismas métricas con Scikit-Learn (para comprobar)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=data.target_names))

## ¿Por qué no basta con la accuracy?

En diagnóstico médico, un **falso negativo** (decir "sano" a un enfermo) es mucho más grave que un falso
positivo. Por eso miramos el **recall** de la clase maligna, no solo la accuracy global. Si las clases
estuvieran desbalanceadas, la accuracy podría ser alta y aun así fallar en la clase importante.

## Resumen

- Para clasificación binaria: salida **sigmoide** + **binary crossentropy**.
- La **matriz de confusión** (VP, VN, FP, FN) es la base de la evaluación.
- De ella salen **accuracy, precision, recall, especificidad y F1**.
- Elige la métrica según el coste de cada tipo de error.

➡️ En el **Notebook 02** haremos clasificación **multiclase** (3 clases) con softmax.